# Options Pricing Showcase

A walkthrough of the `options_pricing` library: closed-form Black-Scholes-Merton, the Cox-Ross-Rubinstein binomial tree, and Monte Carlo simulation with antithetic + control variates. We then cross-validate analytical Greeks against finite-differences, study Monte Carlo convergence empirically, and benchmark the Python implementation against the native C++ engine.

**Sections**
1. Setup
2. Black-Scholes-Merton — closed form, parity check
3. Greeks — analytical vs. finite difference
4. Binomial tree — convergence to BSM, early-exercise premium
5. Monte Carlo — variance reduction and √N convergence
6. Accuracy comparison table across all three methods
7. Exotic options — Asian and barrier via MC
8. Implied-volatility smile (recovered from prices)
9. C++ vs Python benchmark
10. When to use which method — written analysis


## 1. Setup

In [ ]:
import sys, os, math, time
from pathlib import Path

# Allow running the notebook from the repo root without installing the package.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import options_pricing as op
from options_pricing import (
    bs_price, bs_price_vec, bs_greeks, crr_price, crr_greeks, mc_price,
    fd_greeks, implied_vol, asian_price, barrier_price,
    geometric_asian_price, geometric_asian_price_discrete,
)
from options_pricing.monte_carlo import mc_convergence

plt.rcParams.update({
    'figure.figsize': (9, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
np.set_printoptions(suppress=True, precision=6)
print('options_pricing version:', op.__version__)
print('C++ engine loaded:', op.cpp_engine is not None)

## 2. Black-Scholes-Merton

Under the risk-neutral measure $\mathbb{Q}$, the asset $S$ follows
$$dS = (r - q) S\, dt + \sigma S\, dW$$
so $\log S_T \sim \mathcal{N}(\log S + (r-q-\sigma^2/2)T,\ \sigma^2 T)$.

The closed-form European call price is
$$C = S e^{-qT} N(d_1) - K e^{-rT} N(d_2),\quad d_{1,2} = \frac{\log(S/K) + (r - q \pm \sigma^2/2)T}{\sigma\sqrt T}.$$

**Put-call parity** is a model-free arbitrage relation:
$$C - P = S e^{-qT} - K e^{-rT}.$$

In [ ]:
S, K, T, r, sigma, q = 100.0, 100.0, 1.0, 0.05, 0.25, 0.0

call = bs_price(S, K, T, r, sigma, q, 'call')
put  = bs_price(S, K, T, r, sigma, q, 'put')
parity_rhs = S * math.exp(-q*T) - K * math.exp(-r*T)

print(f'Call:           {call:.6f}')
print(f'Put:            {put:.6f}')
print(f'C - P:          {call - put:.6f}')
print(f'S e^-qT - K e^-rT: {parity_rhs:.6f}')
print(f'Parity holds:   {abs((call-put) - parity_rhs) < 1e-10}')

## 3. Greeks — analytical vs. finite difference

Every closed-form Greek should match a finite-difference derivative of `bs_price`. Mismatches indicate either a formula bug or an inappropriate bump size.

In [ ]:
cases = [
    (100, 100, 1.0, 0.05, 0.25, 0.0, 'call'),
    (100, 100, 1.0, 0.05, 0.25, 0.0, 'put'),
    (100, 110, 0.5, 0.03, 0.20, 0.02, 'call'),
    (100,  90, 2.0, 0.04, 0.30, 0.01, 'put'),
    ( 50, 100, 1.0, 0.05, 0.40, 0.00, 'call'),
    (150, 100, 1.0, 0.05, 0.20, 0.00, 'call'),
]
rows = []
for S, K, T, r, sigma, q, opt in cases:
    a = bs_greeks(S, K, T, r, sigma, q, opt)
    f = fd_greeks(bs_price, S, K, T, r, sigma, q, opt)
    rows.append({
        'case': f'S={S} K={K} T={T} σ={sigma} {opt}',
        'Δ_analytical': a.delta, 'Δ_FD': f.delta, 'Δ_err': a.delta - f.delta,
        'Γ_analytical': a.gamma, 'Γ_FD': f.gamma, 'Γ_err': a.gamma - f.gamma,
        'vega_analytical': a.vega, 'vega_FD': f.vega, 'vega_err': a.vega - f.vega,
    })
df = pd.DataFrame(rows).set_index('case')
df.round(6)

In [ ]:
# Greeks across spot — visualize how delta and gamma behave
S_grid = np.linspace(60, 140, 81)
deltas = [bs_greeks(s, 100, 1.0, 0.05, 0.25, 0.0, 'call').delta for s in S_grid]
gammas = [bs_greeks(s, 100, 1.0, 0.05, 0.25, 0.0, 'call').gamma for s in S_grid]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(S_grid, deltas, lw=2)
axes[0].set_xlabel('Spot S'); axes[0].set_ylabel('Delta')
axes[0].set_title('Call delta vs spot (K=100, T=1, σ=0.25, r=5%)')
axes[0].axvline(100, ls='--', color='grey', alpha=0.5)

axes[1].plot(S_grid, gammas, lw=2, color='tab:orange')
axes[1].set_xlabel('Spot S'); axes[1].set_ylabel('Gamma')
axes[1].set_title('Call gamma vs spot (peaks at-the-money)')
axes[1].axvline(100, ls='--', color='grey', alpha=0.5)
plt.tight_layout(); plt.show()

## 4. Binomial tree (Cox-Ross-Rubinstein)

Per step of length $\Delta t = T/n$:
$$u = e^{\sigma\sqrt{\Delta t}},\ d = 1/u,\ p = \frac{e^{(r-q)\Delta t} - d}{u-d}.$$

Build the lattice, take payoffs at the leaves, then backward-induct discounting at $e^{-r\Delta t}$. For American options, max against the intrinsic at every node.

**Convergence**: error is $O(1/n)$. Even-odd oscillation is visible at low $n$ because nodes lie alternately above and below the strike.

In [ ]:
args = (100, 100, 1.0, 0.05, 0.25, 0.0, 'call')
bsm = bs_price(*args)
ns = np.arange(10, 1001, 10)
tree_prices = [crr_price(*args, n_steps=int(n)) for n in ns]

plt.figure(figsize=(10, 4))
plt.plot(ns, tree_prices, lw=1.2, label='CRR binomial')
plt.axhline(bsm, ls='--', color='tab:orange', label=f'BSM = {bsm:.4f}')
plt.xlabel('Tree steps n')
plt.ylabel('Price')
plt.title('CRR price → BSM as n → ∞ (with even-odd oscillation)')
plt.legend(); plt.tight_layout(); plt.show()

print(f'CRR with n=1000: {tree_prices[-1]:.6f} (BSM: {bsm:.6f}, diff: {tree_prices[-1]-bsm:+.6f})')

In [ ]:
# American-put early exercise premium
args = (100, 110, 1.0, 0.05, 0.25, 0.0, 'put')
eu = crr_price(*args, n_steps=500, american=False)
am = crr_price(*args, n_steps=500, american=True)
print(f'European put: {eu:.4f}')
print(f'American put: {am:.4f}')
print(f'Early-exercise premium: {am - eu:+.4f}')

# American call on non-dividend stock: by Merton, no premium → equals European
args = (100, 100, 1.0, 0.05, 0.25, 0.0, 'call')
eu_c = crr_price(*args, n_steps=500, american=False)
am_c = crr_price(*args, n_steps=500, american=True)
print(f'\nAmerican call (no div) - European call: {am_c - eu_c:+.6f}  (≈ 0 expected)')

## 5. Monte Carlo — variance reduction and convergence

Plain MC has standard error $\sigma_{\text{payoff}} / \sqrt{N}$ — error falls as $O(1/\sqrt N)$, so doubling accuracy requires **4× the samples**.

Variance reduction multiplies the constant but does *not* change the rate:
- **Antithetic variates**: pair $Z$ with $-Z$. For monotone payoffs the two are negatively correlated → lower variance than $2N$ independent samples.
- **Control variate**: $\tilde Y = Y - \beta(X - \mathbb{E}[X])$, choosing $X$ correlated with $Y$ but with known expectation. We use $X = e^{-rT} S_T$ with $\mathbb{E}[X] = S e^{-qT}$ — exact under the GBM model.

In [ ]:
args = (100, 100, 1.0, 0.05, 0.25, 0.0, 'call')
bsm = bs_price(*args)
Ns = [1000, 2000, 5000, 10_000, 25_000, 50_000, 100_000, 250_000, 500_000, 1_000_000]

def sweep(antithetic, cv, seed=42):
    return [mc_price(*args, n_paths=N, antithetic=antithetic, control_variate=cv,
                     seed=seed, return_full=True) for N in Ns]

plain  = sweep(False, False)
anti   = sweep(True,  False)
cv     = sweep(False, True)
both   = sweep(True,  True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: price ±2 SE
for label, results, color in [
    ('Plain MC',         plain, 'tab:blue'),
    ('+ Antithetic',     anti,  'tab:green'),
    ('+ Control variate',cv,    'tab:purple'),
    ('Both',             both,  'tab:red'),
]:
    prices = [r.price for r in results]
    ses    = [r.std_error for r in results]
    axes[0].errorbar(Ns, prices, yerr=[2*se for se in ses],
                     marker='o', ms=4, capsize=2, lw=1, label=label, color=color)
axes[0].axhline(bsm, ls='--', color='black', alpha=0.5, label=f'BSM = {bsm:.4f}')
axes[0].set_xscale('log')
axes[0].set_xlabel('Paths N'); axes[0].set_ylabel('Price (±2 SE)')
axes[0].set_title('MC convergence with variance reduction')
axes[0].legend(fontsize=8)

# Right: SE on log-log; slope ≈ -1/2 confirms O(1/√N)
for label, results, color in [
    ('Plain MC',         plain, 'tab:blue'),
    ('+ Antithetic',     anti,  'tab:green'),
    ('+ Control variate',cv,    'tab:purple'),
    ('Both',             both,  'tab:red'),
]:
    ses = [r.std_error for r in results]
    axes[1].loglog(Ns, ses, marker='o', ms=4, label=label, color=color)
axes[1].loglog(Ns, [plain[0].std_error * math.sqrt(Ns[0]) / math.sqrt(N) for N in Ns],
               ls=':', color='black', alpha=0.5, label='1/√N reference')
axes[1].set_xlabel('Paths N'); axes[1].set_ylabel('Standard error')
axes[1].set_title('Slope -½ on log-log = O(1/√N) convergence')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Quantify the variance reduction at N=100k
i = Ns.index(100_000)
print(f"At N = 100,000:")
print(f"  Plain MC SE:                 {plain[i].std_error:.5f}")
print(f"  + Antithetic:                {anti[i].std_error:.5f}   ({plain[i].std_error/anti[i].std_error:.1f}× tighter)")
print(f"  + Control variate:           {cv[i].std_error:.5f}   ({plain[i].std_error/cv[i].std_error:.1f}× tighter)")
print(f"  Both:                        {both[i].std_error:.5f}   ({plain[i].std_error/both[i].std_error:.1f}× tighter)")
print(f"\nA tighter SE by factor k is equivalent to needing k² fewer paths.")

## 6. Accuracy comparison across methods

Same option, three methods. Note:
- BSM is exact under the model (to floating-point); time scales with one CDF evaluation.
- Binomial converges at $O(1/n)$ and runs in $O(n^2)$ memory/work.
- MC converges at $O(1/\sqrt N)$ and is embarrassingly parallel.


In [ ]:
scenarios = [
    ('ATM 1y',    100, 100, 1.00, 0.05, 0.25, 0.0, 'call'),
    ('OTM 6m',    100, 110, 0.50, 0.05, 0.20, 0.0, 'call'),
    ('ITM 2y',    120, 100, 2.00, 0.04, 0.30, 0.02, 'put'),
    ('Hi vol',    100, 100, 1.00, 0.05, 0.60, 0.0, 'call'),
    ('Long-dated',100, 100, 5.00, 0.04, 0.25, 0.0, 'call'),
]

rows = []
for name, S, K, T, r, sigma, q, opt in scenarios:
    args = (S, K, T, r, sigma, q, opt)

    t0 = time.perf_counter()
    p_bs = bs_price(*args)
    t_bs = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    p_crr = crr_price(*args, n_steps=1000)
    t_crr = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    res_mc = mc_price(*args, n_paths=200_000, seed=42, return_full=True)
    t_mc = (time.perf_counter() - t0) * 1000

    rows.append({
        'scenario':     name,
        'BSM':          p_bs,
        'Binomial':     p_crr,
        'MC':           res_mc.price,
        'MC SE':        res_mc.std_error,
        'CRR-BSM err':  p_crr - p_bs,
        'MC-BSM err':   res_mc.price - p_bs,
        't_BSM ms':     t_bs,
        't_CRR ms':     t_crr,
        't_MC ms':      t_mc,
    })
comparison = pd.DataFrame(rows).set_index('scenario')
comparison.round(4)

## 7. Exotic options

Asian and barrier payoffs depend on the **path**, not just $S_T$, so BSM doesn't apply. We price them with MC and use the geometric Asian as a control variate for the arithmetic Asian (they're > 99% correlated).

A subtlety worth noticing: the simulation averages over **discrete** observation dates, so the control's expected value comes from the discrete-sampling geometric closed form (`geometric_asian_price_discrete`) with the same number of observations. The continuous-averaging Kemna-Vorst formula answers a different question — its mean differs visibly at coarse sampling, and a control variate is only valid when its mean is exact for the control *as simulated*.

In [ ]:
S, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.30
n_steps = 50

vanilla = bs_price(S, K, T, r, sigma, 0.0, 'call')
geo_cont = geometric_asian_price(S, K, T, r, sigma, 0.0, 'call')
geo_disc = geometric_asian_price_discrete(S, K, T, r, sigma, 0.0, 'call', n_obs=n_steps)
asian   = asian_price(S, K, T, r, sigma, 0.0, 'call',
                      n_paths=100_000, n_steps=n_steps, seed=42)

print(f'Vanilla European call:                    {vanilla:.4f}')
print(f'Geometric Asian, continuous averaging:    {geo_cont:.4f}')
print(f'Geometric Asian, discrete ({n_steps} obs):       {geo_disc:.4f}   <- the CV reference')
print(f'Arithmetic Asian (MC + control variate):  {asian.price:.4f}  ±{asian.std_error:.5f}')
print(f'\nAveraging lowers effective vol → Asians are cheaper than vanilla.')
print(f'Arithmetic > geometric (AM-GM inequality on lognormals), and')
print(f'discrete > continuous (fewer averaging dates retain more variance).')

In [ ]:
# Barrier in/out parity: knock-out + knock-in = vanilla
S, K, T, r, sigma, B = 100, 100, 1.0, 0.05, 0.25, 120
out = barrier_price(S, K, T, r, sigma, B, 'up-and-out', 0.0, 'call',
                    n_paths=100_000, n_steps=252, seed=42)
inn = barrier_price(S, K, T, r, sigma, B, 'up-and-in',  0.0, 'call',
                    n_paths=100_000, n_steps=252, seed=42)
vanilla = bs_price(S, K, T, r, sigma, 0.0, 'call')

print(f'Up-and-out call:  {out.price:.4f}  ±{out.std_error:.5f}  ({out.extras["fraction_breached"]:.1%} of paths breached)')
print(f'Up-and-in  call:  {inn.price:.4f}  ±{inn.std_error:.5f}')
print(f'Sum:              {out.price + inn.price:.4f}')
print(f'Vanilla:          {vanilla:.4f}')
print(f'Sum - vanilla:    {(out.price + inn.price) - vanilla:+.4f}   (should ≈ 0)')

## 8. Implied-volatility smile

We feed BSM prices back through the IV solver — round-trip should be exact. If you replaced these with *market* prices you'd see the classic skew (downside strikes priced at higher IVs).

In [ ]:
S, T, r = 100, 0.5, 0.04
strikes = np.linspace(70, 130, 25)
# Pretend the market exhibits a skew
true_ivs = 0.20 + 0.15 * np.exp(-((strikes - 100) / 25)**2 * 0.5) - 0.001 * (strikes - 100)

# Vectorized pricing across the whole strike/vol grid in one call
prices = bs_price_vec(S, strikes, T, r, true_ivs, 0.0, 'call')
recovered = [implied_vol(p, S, K, T, r, 0, 'call') for p, K in zip(prices, strikes)]

plt.figure(figsize=(9, 4))
plt.plot(strikes, true_ivs * 100, 'o', label='Input IV (skewed surface)', ms=5)
plt.plot(strikes, np.array(recovered) * 100, '+', label='Recovered IV', ms=10)
plt.xlabel('Strike K'); plt.ylabel('Implied vol (%)')
plt.title('Brent-method IV solver round-trip')
plt.legend(); plt.tight_layout(); plt.show()

max_err = np.max(np.abs(np.array(recovered) - true_ivs))
print(f'Max round-trip error: {max_err:.2e}')

## 9. C++ vs Python benchmark (bonus)

The C++ engine implements the same algorithms. For tight inner loops (binomial, MC) the speedup is significant; for BSM closed form the difference is dominated by Python call overhead.

In [ ]:
if op.cpp_engine is None:
    print('C++ engine not built — skipping benchmark.')
    print('Build it with:  pip install -e ".[dev]"')
else:
    cpp = op.cpp_engine
    args = (100, 100, 1.0, 0.05, 0.25, 0.0, 'call')

    def time_it(fn, *args, n=100, **kwargs):
        t0 = time.perf_counter()
        for _ in range(n):
            fn(*args, **kwargs)
        return (time.perf_counter() - t0) / n * 1000

    py_bs  = time_it(bs_price, *args, n=10_000)
    cpp_bs = time_it(cpp.bs_price, *args, n=10_000)
    print(f'BSM:        Python {py_bs*1000:.1f} µs  vs  C++ {cpp_bs*1000:.1f} µs  ({py_bs/cpp_bs:.1f}× speedup)')

    py_crr  = time_it(crr_price,     *args, n=20, n_steps=1000)
    cpp_crr = time_it(cpp.crr_price, *args, n=20, n_steps=1000, american=False)
    print(f'CRR(1000):  Python {py_crr:.2f} ms  vs  C++ {cpp_crr:.2f} ms     ({py_crr/cpp_crr:.1f}× speedup)')

    py_mc  = time_it(mc_price,      *args, n=5, n_paths=500_000, seed=42)
    cpp_mc = time_it(cpp.mc_price,  *args, n=5, n_paths=500_000, seed=42)
    print(f'MC(500k):   Python {py_mc:.1f} ms   vs  C++ {cpp_mc:.1f} ms     ({py_mc/cpp_mc:.1f}× speedup)')

## 10. When to use which method

All three methods price the same option under the same model — they differ in **what they're best at**.

### Black-Scholes-Merton (closed form)
**Use when**: European exercise, vanilla payoff, lognormal assumption is acceptable.

**Strengths**: instantaneous, exact (to floating-point), analytical Greeks fall out for free. Indispensable as the reference for calibrating models and validating other methods.

**Limitations**: no early exercise, no path-dependence, no stochastic vol or jumps without leaving the model. The constant-vol assumption is a known fiction — see the IV smile in section 8.

### Cox-Ross-Rubinstein binomial tree
**Use when**: early exercise (American options), discrete dividends, simple barriers — anything that needs decisions at intermediate dates but isn't prohibitively high-dimensional.

**Strengths**: handles early exercise naturally (max with intrinsic at every node), Greeks readable from the tree without re-pricing, easy to extend with discrete cash flows.

**Limitations**: $O(n^2)$ work and memory; converges only as $O(1/n)$; oscillates near the strike (lookup *control variate technique* and *Richardson extrapolation* for fixes); curse of dimensionality kicks in past 2-3 underlyings.

### Monte Carlo simulation
**Use when**: path-dependence (Asian, barrier, lookback), multiple correlated underlyings (basket, rainbow), exotic stochastic models, or any payoff that has a clean simulation but a messy or non-existent closed form.

**Strengths**: dimensionality-independent (an $N$-asset basket costs the same per path as a single-asset option, up to the cost of generating correlated noise); trivially parallelizable; tells you its own error via the central limit theorem; variance reduction is a deep toolbox (antithetic, controls, importance sampling, stratification, quasi-MC).

**Limitations**: $O(1/\sqrt N)$ — slow when you want lots of decimal places; Greeks via re-pricing are noisy (use common random numbers, or pathwise/likelihood-ratio estimators); naive American MC fails (look up Longstaff-Schwartz regression).

### Decision shorthand
| Payoff type                            | First choice                  |
|----------------------------------------|-------------------------------|
| European vanilla, single asset         | Black-Scholes                 |
| American vanilla, single asset         | Binomial (or PDE)             |
| Asian / lookback / barrier             | Monte Carlo (+ control variate)|
| Basket / multi-asset / correlated      | Monte Carlo                   |
| Discrete dividends, deterministic      | Binomial                      |
| Calibration (need a fast inner loop)   | Black-Scholes                 |
